In [13]:
import pandas as pd
import numpy as np

# 1. Cargar la base de datos (cambia la ruta si quieres probar la BBDD 2)
ruta_bbdd = '../../data/dataset_features_temperatura.csv'

try:
    df = pd.read_csv(ruta_bbdd)
    
    # 2. Seleccionamos solo las columnas numéricas para evitar errores con nombres de archivos
    df_numerico = df.select_dtypes(include=[np.number])
    
    # 3. Calculamos la correlación de Pearson de todo contra la 'temperature'
    correlaciones = df_numerico.corr()['temperature'].drop('temperature')
    
    # 4. Ordenamos por el valor absoluto (nos importan las que están cerca de 1 o de -1)
    mejores_variables = correlaciones.abs().sort_values(ascending=False)
    
    print("🏆 TOP 15 CARACTERÍSTICAS MÁS CORRELACIONADAS CON LA TEMPERATURA 🏆")
    print(f"Analizando: {ruta_bbdd}\n")
    print(f"{'CARACTERÍSTICA':<30} | {'CORRELACIÓN':<12} | {'TENDENCIA'}")
    print("-" * 65)
    
    for col in mejores_variables.head(15).index:
        corr_val = correlaciones[col]
        # Clasificamos si va hacia arriba o hacia abajo
        tendencia = "📈 Sube con el calor" if corr_val > 0 else "📉 Baja con el calor"
        
        # Formateamos para que quede bonito
        print(f"{col:<30} | {corr_val:>10.4f}   | {tendencia}")

except Exception as e:
    print(f"❌ Error al procesar el archivo: {e}")

🏆 TOP 15 CARACTERÍSTICAS MÁS CORRELACIONADAS CON LA TEMPERATURA 🏆
Analizando: ../../data/dataset_features_temperatura.csv

CARACTERÍSTICA                 | CORRELACIÓN  | TENDENCIA
-----------------------------------------------------------------
temperature_folder             |     1.0000   | 📈 Sube con el calor
mag_min                        |    -0.3616   | 📉 Baja con el calor
svd_sigma_2                    |     0.3586   | 📈 Sube con el calor
svd_sigma_ratio                |    -0.2990   | 📉 Baja con el calor
mag_var                        |     0.2973   | 📈 Sube con el calor
phase_jump_m_max_abs           |     0.2856   | 📈 Sube con el calor
mag_energy_total               |     0.2806   | 📈 Sube con el calor
pdp_total_energy               |     0.2806   | 📈 Sube con el calor
mag_dynamic_range_db           |     0.2806   | 📈 Sube con el calor
phase_jump_k_max_abs           |     0.2750   | 📈 Sube con el calor
svd_spectral_flatness          |     0.2696   | 📈 Sube con el calor
pdp_l

In [14]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

# =============================================================================
# 🎯 1. CONFIGURACIÓN: CAMBIA ESTO PARA EXPLORAR OTRAS VARIABLES
# =============================================================================
CARACTERISTICA_A_ESTUDIAR = 'svd_sigma_2' # <-- Escribe aquí la variable a probar

print(f"🔬 EXPLORANDO CARACTERÍSTICA: {CARACTERISTICA_A_ESTUDIAR.upper()} 🔬\n")
print("=" * 60)

# =============================================================================
# 2. CONECTAR FUNCIONES Y RUTAS
# =============================================================================
ruta_hugo = Path("../../Hugo").resolve()
if str(ruta_hugo) not in sys.path:
    sys.path.insert(0, str(ruta_hugo))

try:
    from channel_estimator import compute_channel_matrix_from_iq_paths
    from channel_features_complete import extract_channel_matrix_features
except ImportError:
    print("❌ ERROR: No se han podido importar las funciones de Hugo.")
    exit()

RUTA_YAML = Path("../../data/Modulator.yaml")

# Función auxiliar para aplicar las 3 normalizaciones a las medias
def aplicar_normalizaciones(df_medias, col_name):
    # Cogemos los valores crudos para escalar su tendencia
    datos = df_medias[[col_name]].values
    
    df_res = df_medias.copy()
    df_res[f'MinMax(0-1)'] = MinMaxScaler().fit_transform(datos)
    df_res[f'Standard(Z)'] = StandardScaler().fit_transform(datos)
    df_res[f'Robust'] = RobustScaler().fit_transform(datos)
    
    # Redondeamos un poco para que la tabla sea legible en consola
    return df_res.round(4)

# =============================================================================
# 3. ANÁLISIS DE LAS BBDD ANTIGUAS (CSV)
# =============================================================================
rutas_antiguas = {
    "BBDD 1 (26k - Antigua)": "../dataset_features_temperatura.csv",
    "BBDD 2 (32k - Alternativa)": "../../data/dataset_features_temperatura.csv"
}

for nombre, ruta in rutas_antiguas.items():
    try:
        df = pd.read_csv(ruta)
        if CARACTERISTICA_A_ESTUDIAR in df.columns:
            # Agrupar por temperatura y sacar la media
            df_medias = df[['temperature', CARACTERISTICA_A_ESTUDIAR]].groupby('temperature').mean()
            
            # Normalizar
            df_final = aplicar_normalizaciones(df_medias, CARACTERISTICA_A_ESTUDIAR)
            
            print(f"\n📊 --- {nombre} --- 📊")
            print(df_final.to_string())
        else:
            print(f"\n⚠️ La variable '{CARACTERISTICA_A_ESTUDIAR}' no existe en {nombre}.")
    except Exception as e:
        print(f"\n❌ Error al cargar {nombre}: {e}")

# =============================================================================
# 4. ANÁLISIS DE LAS BBDD NUEVAS EN VIVO (.BIN)
# =============================================================================
materiales = ['carton', 'cristal', 'plastico']
condiciones = {'frio': 15.0, 'templado': 40.0, 'caliente': 90.0}
muestras = ['1', '2'] # Suponiendo que hay muestra 1 y 2 por cada vaso

print("\n\n" + "=" * 60)
print("📡 PROCESANDO GRABACIONES EN VIVO (CARTÓN, CRISTAL Y PLÁSTICO)...")
print("=" * 60)

for material in materiales:
    datos_material = []
    ruta_base = Path(f"../../data/{material}")
    
    for cond_str, temp_val in condiciones.items():
        for m in muestras:
            tx_path = ruta_base / cond_str / f"iq_tx_{m}.bin"
            rx_path = ruta_base / cond_str / f"iq_rx_{m}.bin"
            
            if not tx_path.exists() or not rx_path.exists():
                continue
                
            try:
                H = compute_channel_matrix_from_iq_paths(
                    tx_path=tx_path, rx_path=rx_path, yaml_path=RUTA_YAML,
                    fftshift=False, normalize_fft=False, trim_to_complete_frames=True,
                    output_order="mk", verbose=False
                )
                features = extract_channel_matrix_features(H, input_order="mk")
                
                if CARACTERISTICA_A_ESTUDIAR in features:
                    datos_material.append({
                        'temperature': temp_val,
                        CARACTERISTICA_A_ESTUDIAR: features[CARACTERISTICA_A_ESTUDIAR]
                    })
            except Exception as e:
                pass # Ignoramos errores de lectura individuales para no manchar la consola

    if len(datos_material) > 0:
        df_mat = pd.DataFrame(datos_material)
        df_medias_mat = df_mat.groupby('temperature').mean()
        df_final_mat = aplicar_normalizaciones(df_medias_mat, CARACTERISTICA_A_ESTUDIAR)
        
        print(f"\n🧪 --- MATERIAL: {material.upper()} --- 🧪")
        print(df_final_mat.to_string())
    else:
        print(f"\n⚠️ No se encontraron datos o falló la extracción para el material: {material}")

🔬 EXPLORANDO CARACTERÍSTICA: SVD_SIGMA_2 🔬


📊 --- BBDD 1 (26k - Antigua) --- 📊
             svd_sigma_2  MinMax(0-1)  Standard(Z)  Robust
temperature                                               
15.0             32.1052       0.5425       0.1877  0.0000
19.0             32.4988       0.5671       0.2795  0.1319
23.0             39.4461       1.0000       1.8993  2.4610
28.0             29.7311       0.3946      -0.3659 -0.7959
33.0             30.2948       0.4297      -0.2344 -0.6069
40.0             23.3991       0.0000      -1.8423 -2.9187
45.0             24.0667       0.0416      -1.6866 -2.6948
52.0             32.2567       0.5520       0.2230  0.0508
58.0             28.1404       0.2955      -0.7368 -1.3292
62.0             30.9032       0.4676      -0.0926 -0.4030
70.0             34.9162       0.7177       0.8431  0.9424
80.0             36.4308       0.8121       1.1963  1.4501
90.0             32.7140       0.5805       0.3296  0.2041

📊 --- BBDD 2 (32k - Alternativa) -